# Dia 18 — Quem registrou? 🌐

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-18-investigacao-de-dominio](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-18-investigacao-de-dominio)

## 🎬 Missão

> A Jurema chega com uma pasta de papel (sim, papel).
> — Ofício respondido pelo registro. Três domínios aparecem nas denúncias:
> `pixvelox.example`, `suporte-pixvelox.example` e `cashback-pix-oficial.example`.
> Cada um em nome de uma pessoa diferente.
>
> — Laranjas — resume Bento. — Nome de dono não vale nada.
>
> A Mestra Iara folheia as saídas de `whois` e `dig` que a equipe salvou.
> — Dono se troca. Encanamento, não. Quem configura o DNS, quem é o contato
> técnico, qual e-mail recebe os avisos… isso o golpista reaproveita. Descubra
> quais domínios são **da mesma mão** — e o que eles deixaram escrito no DNS.

**Sua tarefa:** correlacionar os whois e os registros DNS salvos em
`desafio/`, descobrir quais domínios pertencem ao mesmo operador e montar a
flag `VAZANTE{...}` com os registros TXT deles.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### whois: a "certidão" de um domínio

```text
Todo domínio é registrado por alguém, numa data, num registro (o .br é do
registro.br / NIC.br). O `whois` pergunta isso ao servidor do registro.
No .br aparecem: dono (owner), contatos (owner-c, tech-c), servidores de
nome (nserver), datas de criação/alteração/expiração.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "whois registro.br"
precisa whois
if online && whois registro.br > whois_real.txt 2>/dev/null && [ -s whois_real.txt ]; then
  grep -v '^%' whois_real.txt | grep -v '^$' | head -n 25
else
  alerta "offline: usando cópia salva (formato idêntico ao do registro.br)"
  grep -v '^%' desafio/whois/pixvelox.example.txt | grep -v '^$'
fi
rm -f whois_real.txt

### Filtrando o whois com grep

```text
whois devolve muito texto. grep separa o que interessa.
Domínios .com usam "Creation Date"; o registro.br usa "created".
  -i   ignora maiúsculas/minúsculas      -E  vários padrões com |
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "Só as datas e os contatos"
precisa whois
if online && whois example.com > whois_com.txt 2>/dev/null && [ -s whois_com.txt ]; then
  echo "--- example.com (formato internacional) ---"
  grep -i 'Creation' whois_com.txt | head -n 2
fi
rm -f whois_com.txt
echo "--- pixvelox.example (formato registro.br) ---"
grep -E '^(created|changed|expires):' desafio/whois/pixvelox.example.txt
grep -E '^(owner|owner-c|tech-c|e-mail):' desafio/whois/pixvelox.example.txt

### dig: perguntando ao DNS

```text
DNS é a "lista telefônica" da internet: traduz nome em endereço e guarda
outras informações. Tipos de registro mais úteis em OSINT:
  A / AAAA   endereço IPv4 / IPv6 do site
  MX         servidor que recebe os e-mails do domínio
  NS         servidores de nome (quem "responde" pelo domínio)
  TXT        textos livres: SPF, verificações do Google/Microsoft...
+short mostra só a resposta, sem cabeçalhos.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "dig example.com e registro.br"
precisa dig dnsutils
if online && dig +short example.com A >/dev/null 2>&1 && [ -n "$(dig +short example.com A)" ]; then
  dig example.com                  # saída completa
  echo "--- só as respostas (+short) ---"
  dig +short example.com A
  dig +short registro.br MX
  dig +short registro.br NS
  dig +short registro.br TXT | head -n 5
else
  alerta "offline: usando cópia salva"
  cat desafio/dig_pixvelox.example.txt
fi

### Lendo a saída do dig

```text
Na saída completa, a parte que importa é a ANSWER SECTION:
  nome.   TTL   classe   tipo   valor
TTL = quantos segundos a resposta pode ficar guardada em cache.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "A ANSWER SECTION"
grep -A 2 'ANSWER SECTION' desafio/dig_pixvelox.example.txt
echo "--- registros salvos dos domínios do caso (+noall +answer) ---"
cat desafio/dns/pixvelox.example.txt

### Separando por tipo com awk

```text
Cada linha tem 5 colunas: nome, TTL, classe, TIPO, valor. awk filtra a 4ª.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "Registros por tipo"
echo "--- MX ---";  awk '$4 == "MX"'  desafio/dns/pixvelox.example.txt
echo "--- NS ---";  awk '$4 == "NS" {print $5}' desafio/dns/pixvelox.example.txt
echo "--- TXT ---"; awk -F'\t' '$4 == "TXT" {print $5}' desafio/dns/pixvelox.example.txt

### host e nslookup: os primos do dig

```text
`host` dá respostas curtas e legíveis; `nslookup` é o mais antigo (existe
até no Windows). Os dois vêm no mesmo pacote do dig.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "host e nslookup"
precisa host bind9-host
precisa nslookup dnsutils
if online && host example.com >/dev/null 2>&1; then
  host example.com
  host -t MX registro.br
  nslookup -type=NS registro.br | tail -n +4
else
  alerta "offline: só mostrando os comandos"
  mostrar "host example.com"
  mostrar "host -t MX registro.br"
  mostrar "nslookup -type=NS registro.br"
fi

### DNS reverso: de IP para nome

```text
dig -x IP pergunta "qual nome está ligado a este IP?" (registro PTR).
Em hospedagens compartilhadas, revela o provedor.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "DNS reverso"
precisa dig dnsutils
if online && [ -n "$(dig +short -x 8.8.8.8)" ]; then
  dig +short -x 8.8.8.8
else
  alerta "offline: só mostrando o comando"
  mostrar "dig +short -x 8.8.8.8    # -> dns.google."
fi

### Correlação: o que dois domínios têm em comum?

```text
Golpistas trocam de "laranja" no campo owner, mas costumam reaproveitar
infraestrutura: o mesmo técnico, o mesmo e-mail, os mesmos servidores DNS.
Juntamos os campos de todos os whois e contamos repetições.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "Campos repetidos entre os domínios do caso"
for arq in desafio/whois/*.txt; do
  dominio=$(basename "$arq" .txt)
  grep -E '^(tech-c|nserver|e-mail):' "$arq" | sed "s|^|$dominio  |"
done | sort -k2 > campos.txt
cat campos.txt
echo "--- valores que aparecem em MAIS DE UM domínio ---"
awk '{print $2, $3}' campos.txt | sort | uniq -c | awk '$1 > 1'
rm campos.txt

### Quais domínios estão ligados pelo mesmo técnico?

```text
Achamos o handle repetido (tech-c) e listamos os domínios que o usam.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "Domínios do mesmo tech-c"
grep -l '^tech-c: *MAALT12' desafio/whois/*.txt | xargs -n1 basename | sed 's/\.txt$//'
echo "--- e o e-mail desse contato ---"
grep -A 2 '^nic-hdl-br: *MAALT12' desafio/whois/pixvelox.example.txt | grep 'e-mail'

### Netlas host (opcional, precisa de API key)

```text
O Netlas junta whois + DNS + servidores num lugar só.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "netlas host"
mostrar "pip install netlas"
mostrar "netlas savekey SUA_CHAVE"
mostrar "netlas host registro.br"
if online && [ -n "${NETLAS_API_KEY:-}" ] && command -v netlas >/dev/null 2>&1; then
  netlas host -a "$NETLAS_API_KEY" registro.br | head -n 30
else
  dica "Sem NETLAS_API_KEY (ou offline): pulando."
fi

### Versão brazuca: RDAP do registro.br

```text
RDAP é o "whois moderno": mesma informação, mas em JSON (bom para jq).
O registro.br tem RDAP público em https://rdap.registro.br/domain/<dominio>
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && source "$L4O_HOME/lib/comum.sh"
titulo "RDAP do registro.br"
precisa jq
if online && curl -s -m 15 https://rdap.registro.br/domain/registro.br -o rdap.json && jq -e . rdap.json >/dev/null 2>&1; then
  jq -r '.ldhName, (.events[] | "\(.eventAction): \(.eventDate)")' rdap.json
  jq -r '.nameservers[].ldhName' rdap.json
else
  alerta "offline: só mostrando o comando"
  mostrar "curl -s https://rdap.registro.br/domain/registro.br | jq '.events'"
fi
rm -f rdap.json

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-18-investigacao-de-dominio" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)